# Goal:
For new changed data, `post_data_v3.json`, run the **numerical_only** model and then do the **error taxonomy**. And then if applicable, do the feature engineering

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random

In [2]:
random.seed(42)
with open("../../../data/post_quality/post_data_v3.json","r",encoding="utf-8") as f:
    raw_data = json.load(f)


random.shuffle(raw_data)

df = pd.DataFrame(raw_data)
df

,id,title,body,code,label,domain,source
0,47,Why are genetic differences in appearance and ...,"I mean, different human races have lived in di...",,1,priviledge,reddit
1,19,I changed a road sign to make my commute easie...,On my daily commute there was very inconvenien...,,0,personal-story,reddit
2,37,What’s a skill that sounds boring but is actua...,"Some skills don’t sound exciting at first, but...",,2,Thought-question,reddit
3,9,I can post!,So...\n😭💔😶‍🌫️🥱😮‍💨🙄😒😓😥☹️\nI'm tired.,,0,general-statement,reddit
4,8,My biggest fear is Alzheimer’s,"Not for me, for my mom.\nI don’t think I could...",,0,personal-story,reddit
...,...,...,...,...,...,...,...
70,29,I’m (27f) 80% sure my roommate(30m) wants to s...,"I wanna sleep with him too, honestly he’s tota...",,1,personal-story,reddit
71,32,"Vegas didn’t evolve, it got murdered by MGM an...","Old Vegas: mob skimmed the drop, but drinks we...",,1,personal-belief,reddit
72,36,Beginner Next.js Authentication Project — Look...,Hi everyone 👋\n\nI’m currently learning Next.j...,,1,software-engineering,reddit
73,4,Say something in a language other than English...,,,0,general-question,reddit


# Getting FE_v1 features

In [3]:
import numpy as np
import pandas as pd

import re

FIRST_PERSON = {"i", "me", "my", "mine", "we", "us", "our", "ours"}

def first_person_ratio(text):
    if not isinstance(text, str):
        return 0.0

    words = re.findall(r"\b\w+\b", text.lower())
    if not words:
        return 0.0

    fp_count = sum(1 for w in words if w in FIRST_PERSON)
    return fp_count / len(words)




def build_fe1_features(
    df: pd.DataFrame,
    drop_features: list[str] | None = None,
):
    """
    Feature Engineering v1 (numerical + structural features)

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    drop_features : list[str]
        Columns to drop AFTER feature creation

    Returns
    -------
    pd.DataFrame
        Transformed dataframe with FE1 features
    """

    df = df.copy()

    # -----------------------------
    # 1. Clean NaNs (important)
    # -----------------------------
    text_cols = ["title", "body", "code"]

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("")

    # -----------------------------
    # 2. Base text column
    # -----------------------------
    df["text"] = df.get("title", "") + "\n\n" + df.get("body", "")

    # -----------------------------
    # 3. Length-based features
    # -----------------------------
    df["title_len"] = df["title"].str.len()
    df["body_len"] = df["body"].str.len()
    df["total_len"] = df["text"].str.len()

    df["body_to_title_ratio"] = df["body_len"] / (df["title_len"] + 1)

    # -----------------------------
    # 4. Paragraph + punctuation
    # -----------------------------
    df["num_para"] = df["body"].str.count("\n") + 1
    df["num_questions"] = df["text"].str.count(r"\?")
    df["num_exclamations"] = df["text"].str.count("!")

    df["questions_per_100_words"] = (
        df["num_questions"] / (df["text"].str.split().str.len() / 100)
    )

    # -----------------------------
    # 5. Boolean flags
    # -----------------------------
    df["has_multiple_paras"] = (df["num_para"] > 1).astype(int)
    df["has_questions"] = (df["num_questions"] > 0).astype(int)
    df["has_exclamations"] = (df["num_exclamations"] > 0).astype(int)

    # -----------------------------
    # 6. Log transforms
    # -----------------------------
    df["log_num_para"] = np.log1p(df["num_para"])

    # -----------------------------
    # 7. Word statistics
    # -----------------------------
    df["word_count"] = df["text"].str.split().str.len()

    df["avg_word_len"] = df["text"].str.split().apply(
        lambda x: np.mean([len(w) for w in x]) if len(x) else 0
    )

    df["unique_word_ratio"] = (
        df["text"]
        .str.lower()
        .str.split()
        .apply(lambda x: len(set(x)) if len(x) else 0)
        / df["word_count"].replace(0, 1)
    )

    # -----------------------------
    # 8. Code presence
    # -----------------------------
    if "code" in df.columns:
        df["has_code"] = (df["code"].str.len() > 0).astype(int)
    else:
        df["has_code"] = 0

    # -----------------------------
    # 9. Log-ratio engineered features
    # -----------------------------
    df["log_btt"] = np.log1p(df["body_len"]) - np.log1p(df["title_len"])
    df["log_btt_x_has_questions"] = df["log_btt"] * df["has_questions"]

    df["question_density_per_para"] = (
        df["num_questions"] / (df["body"].str.count("\n") + 1)
    )


    # -----------------------------
    # 10. First person pronoun feature
    # -----------------------------
    df['first_person_ratio'] = df['text'].apply(first_person_ratio)


    # -----------------------------
    # 11. Drop intermediate columns safely
    # -----------------------------
    auto_drop = [
        "total_len",
        "num_para",
        "num_questions",
        "has_questions",
        "body_to_title_ratio",
        "id",
        "domain",
        "source"
    ]

    df = df.drop(columns=[c for c in auto_drop if c in df.columns])

    # -----------------------------
    # 12. User-requested drops
    # -----------------------------
    if drop_features:
        df = df.drop(columns=[c for c in drop_features if c in df.columns])

    return df


In [4]:
df = build_fe1_features(df)

In [5]:
df.columns

Index(['title', 'body', 'code', 'label', 'text', 'title_len', 'body_len',
       'num_exclamations', 'questions_per_100_words', 'has_multiple_paras',
       'has_exclamations', 'log_num_para', 'word_count', 'avg_word_len',
       'unique_word_ratio', 'has_code', 'log_btt', 'log_btt_x_has_questions',
       'question_density_per_para', 'first_person_ratio'],
      dtype='object')

In [6]:
df

,title,body,code,label,text,title_len,body_len,num_exclamations,questions_per_100_words,has_multiple_paras,has_exclamations,log_num_para,word_count,avg_word_len,unique_word_ratio,has_code,log_btt,log_btt_x_has_questions,question_density_per_para,first_person_ratio
0,Why are genetic differences in appearance and ...,"I mean, different human races have lived in di...",,1,Why are genetic differences in appearance and ...,188,210,0,3.174603,0,0,0.693147,63,5.349206,0.777778,0,0.110111,0.110111,2.0,0.015625
1,I changed a road sign to make my commute easie...,On my daily commute there was very inconvenien...,,0,I changed a road sign to make my commute easie...,61,1185,0,0.000000,1,0,1.791759,255,3.894118,0.552941,0,2.951207,0.000000,0.0,0.065385
2,What’s a skill that sounds boring but is actua...,"Some skills don’t sound exciting at first, but...",,2,What’s a skill that sounds boring but is actua...,58,309,0,1.612903,0,0,0.693147,62,4.951613,0.806452,0,1.659035,1.659035,1.0,0.014925
3,I can post!,So...\n😭💔😶‍🌫️🥱😮‍💨🙄😒😓😥☹️\nI'm tired.,,0,I can post!\n\nSo...\n😭💔😶‍🌫️🥱😮‍💨🙄😒😓😥☹️\nI'm ti...,11,33,1,0.000000,1,1,1.386294,7,5.571429,1.000000,0,1.041454,0.000000,0.0,0.285714
4,My biggest fear is Alzheimer’s,"Not for me, for my mom.\nI don’t think I could...",,0,"My biggest fear is Alzheimer’s\n\nNot for me, ...",30,133,0,0.000000,1,0,1.098612,33,4.000000,0.787879,0,1.463853,0.000000,0.0,0.142857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,I’m (27f) 80% sure my roommate(30m) wants to s...,"I wanna sleep with him too, honestly he’s tota...",,1,I’m (27f) 80% sure my roommate(30m) wants to s...,58,762,0,0.000000,0,0,0.693147,157,4.235669,0.719745,0,2.559721,0.000000,0.0,0.120482
71,"Vegas didn’t evolve, it got murdered by MGM an...","Old Vegas: mob skimmed the drop, but drinks we...",,1,"Vegas didn’t evolve, it got murdered by MGM an...",78,889,0,0.000000,1,0,1.791759,179,4.413408,0.698324,0,2.421774,0.000000,0.0,0.005435
72,Beginner Next.js Authentication Project — Look...,Hi everyone 👋\n\nI’m currently learning Next.j...,,1,Beginner Next.js Authentication Project — Look...,83,957,1,0.000000,1,1,3.295837,147,6.034014,0.768707,0,2.434031,0.000000,0.0,0.044872
73,Say something in a language other than English...,,,0,Say something in a language other than English...,92,0,0,0.000000,0,0,0.693147,17,4.470588,0.941176,0,-4.532599,-0.000000,0.0,0.000000


# Helper functions for error taxonomy

In [7]:
## Let's build a helper function to make error_df
import pandas as pd

def build_error_df(df_test, y_true, y_pred, model_name):
    out = df_test.copy()
    out['true_label'] = y_true
    out['pred_label'] = y_pred
    out['correct'] = y_true == y_pred
    out['model'] = model_name
    return out

In [8]:
## Generic function to split df in same test and train
from sklearn.model_selection import train_test_split


def split_df(
    df,
    label_col,
    test_size=0.2,
    random_state=52,
    stratify=True
):
    """
    Splits dataframe into train and test sets.

    Returns
    -------
    df_train : pd.DataFrame
    df_test : pd.DataFrame
    """

    df = df.reset_index(drop=True)

    stratify_col = df[label_col] if stratify else None

    train_idx, test_idx = train_test_split(
        df.index,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_col
    )

    df_train = df.loc[train_idx]
    df_test = df.loc[test_idx]

    return df_train, df_test


In [9]:
## Running numerical only model
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score


def run_numerical_model(
    df_train,
    df_test,
    label_col,
    num_cols=None,
    model=None,
    scaler=None,
    model_name="numerical_only"
):
    """
    Trains and evaluates a numerical-only classification model.

    Returns
    -------
    dict with:
        - model
        - scaler
        - classification_report
        - errors_df
        - y_pred
    """

    if num_cols is None:
        num_cols = df_train.select_dtypes(include="number").columns.tolist()
        num_cols.remove(label_col)

    if scaler is None:
        scaler = StandardScaler()

    if model is None:
        model = LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )

    X_train = scaler.fit_transform(df_train[num_cols])
    X_test = scaler.transform(df_test[num_cols])

    y_train = df_train[label_col]
    y_test = df_test[label_col]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(y_test, y_pred)

    errors_df = build_error_df(
        df_test=df_test,
        y_true=y_test,
        y_pred=y_pred,
        model_name=model_name
    )

    errors_df['error_type'] = ''
    errors_df['error_subtype'] = ''
    errors_df['notes'] = ''

    return {
        "model": model,
        "scaler": scaler,
        "classification_report": report,
        "errors_df": errors_df,
        "accuracy":accuracy_score(y_test,y_pred),
        "y_pred": y_pred
    }


# Helper function for validating error taxonomy

In [10]:
## Function to get error across k-folds
import pandas as pd
from sklearn.model_selection import StratifiedKFold

def run_k_fold_error_analysis(df:pd.DataFrame, n_splits=5, label_col='label'):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )
    
    all_errors = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df[label_col])):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        
        result = run_numerical_model(
            train_df,
            test_df,
            label_col=label_col
        )
        
        errors_df = result['errors_df'].copy()
        
        errors_df['fold'] = fold
        
        all_errors.append(errors_df)
    
    errors_all = pd.concat(all_errors).reset_index(drop=True)
    
    errors_only = errors_all[errors_all['correct'] == False]
    
    errors_all['error_type'] = 'N/A'  
    errors_all['error_subtype'] = 'N/A'  
    errors_all['notes'] = ''  
    
    true_label_counts = errors_only['true_label'].value_counts()
    pred_label_counts = errors_only['pred_label'].value_counts()
    error_group_counts = errors_only.groupby(['true_label', 'pred_label']).size()
    
    return {
        'errors_all': errors_all,
        'errors_only': errors_only,
        'true_label_counts': true_label_counts,
        'pred_label_counts': pred_label_counts,
        'error_group_counts': error_group_counts
    }


# Helper function for validating feature impact

In [11]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd


def evaluate_model_with_feature_and_changes(
    df,
    target_label,
    feature_to_check,
    n_splits=5,
    random_state=42,
    text_cols=("title", "body")
):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    acc_with_feature = []
    acc_without_feature = []
    open_acc_with = []
    open_acc_without = []

    changed_rows_all_folds = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df, df[target_label])):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        # -----------------------
        # WITH FEATURE
        # -----------------------
        result_with = run_numerical_model(train_df, test_df, target_label)
        acc_with_feature.append(result_with["accuracy"])

        errors_with = result_with["errors_df"]

        # -----------------------
        # WITHOUT FEATURE
        # -----------------------
        train_df_no = train_df.drop(columns=[feature_to_check], errors="ignore")
        test_df_no = test_df.drop(columns=[feature_to_check], errors="ignore")

        result_without = run_numerical_model(train_df_no, test_df_no, target_label)
        acc_without_feature.append(result_without["accuracy"])

        errors_without = result_without["errors_df"]

        # -----------------------
        # OPEN-ENDED accuracy (your existing logic)
        # -----------------------
        open_test = test_df[test_df[feature_to_check] == 1]

        if len(open_test) > 0:
            wrong_open_with = errors_with[
                (~errors_with["correct"]) &
                (errors_with.index.isin(open_test.index))
            ].shape[0]

            wrong_open_without = errors_without[
                (~errors_without["correct"]) &
                (errors_without.index.isin(open_test.index))
            ].shape[0]

            open_acc_with.append(1 - wrong_open_with / len(open_test))
            open_acc_without.append(1 - wrong_open_without / len(open_test))

        # =======================
        # FIND CHANGED PREDICTIONS
        # =======================

        merged = (
            test_df
            .loc[:, list(text_cols) + [feature_to_check]]
            .copy()
        )

        merged["true_label"] = df.loc[test_idx, target_label]
        merged["pred_without_feature"] = errors_without["pred_label"]
        merged["pred_with_feature"] = errors_with["pred_label"]

        # Keep only rows where prediction changed
        changed = merged[
            merged["pred_with_feature"] != merged["pred_without_feature"]
        ].copy()

        if not changed.empty:
            changed["fold"] = fold
            changed_rows_all_folds.append(changed)

    # Combine all folds
    changed_df = (
        pd.concat(changed_rows_all_folds, axis=0)
        if changed_rows_all_folds
        else pd.DataFrame()
    )

    results = {
        "overall_accuracy_with_feature": np.mean(acc_with_feature),
        "overall_accuracy_without_feature": np.mean(acc_without_feature),
        "open_ended_accuracy_with_feature": np.mean(open_acc_with),
        "open_ended_accuracy_without_feature": np.mean(open_acc_without),
        "changed_predictions_df": changed_df
    }

    return results


In [12]:
df_train, df_test = split_df(df,'label')
result = run_numerical_model(df_train,df_test,'label')
df_train.shape,df_test.shape

((60, 20), (15, 20))

In [13]:
all_result = result['errors_df']
error_only = all_result[all_result['correct'] == False].copy()
error_only

,title,body,code,label,text,title_len,body_len,num_exclamations,questions_per_100_words,has_multiple_paras,...,log_btt_x_has_questions,question_density_per_para,first_person_ratio,true_label,pred_label,correct,model,error_type,error_subtype,notes
51,Living in a foreign country is taking its tow,"I love the country I choose to live, since it ...",,1,Living in a foreign country is taking its tow\...,45,285,0,0.000000,1,...,0.000000,0.0,0.032258,1,0,False,numerical_only,,,
34,How would you feel if your country banned Burk...,,,1,How would you feel if your country banned Burk...,73,0,0,7.692308,0,...,-4.304065,1.0,0.000000,1,0,False,numerical_only,,,
45,Can't afford to live and I've exhausted all ot...,So I've been sitting here for the past three d...,,0,Can't afford to live and I've exhausted all ot...,57,333,0,2.702703,0,...,1.750698,2.0,0.137500,0,1,False,numerical_only,,,
22,I got called a fascist today.,So I was on tiktok and a post came about on a ...,,2,I got called a fascist today.\n\nSo I was on t...,29,531,0,2.564103,0,...,2.875446,3.0,0.095652,2,1,False,numerical_only,,,
74,Add a festive snow effect this Christmas with ...,Hello r/reactjs!\n\nSprinkling some snow acros...,import { SnowOverlay } from 'react-snow-overla...,0,Add a festive snow effect this Christmas with ...,68,502,2,0.000000,1,...,0.000000,0.0,0.037736,0,1,False,numerical_only,,,
66,19F Exhausted New mom not much help from parents.,"Hi i'm a 19f new mom, and I gave birth to my d...",,1,19F Exhausted New mom not much help from paren...,49,1093,0,0.000000,1,...,0.000000,0.0,0.094170,1,0,False,numerical_only,,,
41,what would the modern day versions of the Gree...,,,1,what would the modern day versions of the Gree...,61,0,0,8.333333,0,...,-4.127134,1.0,0.000000,1,0,False,numerical_only,,,
1,I changed a road sign to make my commute easie...,On my daily commute there was very inconvenien...,,0,I changed a road sign to make my commute easie...,61,1185,0,0.000000,1,...,0.000000,0.0,0.065385,0,1,False,numerical_only,,,


In [14]:
error_only.iloc[0, error_only.columns.get_loc('error_type')] = 'D'
error_only.iloc[0, error_only.columns.get_loc('error_subtype')] = 'Pragmatic Intent Gap'
error_only.iloc[0, error_only.columns.get_loc('notes')] = (
    'The post is an experiential reflection intended to invite shared lived experiences, '
    'despite lacking an explicit question. The model over-relies on surface cues such as '
    'the absence of a question mark and misclassifies the post as low quality. Correct '
    'labeling depends on understanding the author’s pragmatic intent to elicit discussion, '
    'which is beyond the capability of numerical or structural features.'
)


In [15]:
error_only.iloc[1, error_only.columns.get_loc('error_type')] = 'D'
error_only.iloc[1, error_only.columns.get_loc('error_subtype')] = 'Structural Bias Masking Intent'
error_only.iloc[1, error_only.columns.get_loc('notes')] = (
    'The model systematically undervalues title-only open-ended questions, downgrading them '
    'due to missing body content. Despite the brevity, the post is conceptually open-ended and '
    'discussion-capable, likely to elicit diverse opinions. The error arises from structural '
    'bias toward post length rather than an inability to generalize from data.'
)
error_only.iloc[2, error_only.columns.get_loc('error_type')] = 'B'
error_only.iloc[2, error_only.columns.get_loc('error_subtype')] = 'Engagement Proxy Mismatch'
error_only.iloc[2, error_only.columns.get_loc('notes')] = (
    'The model overweights emotional intensity and narrative length as proxies for discussion '
    'quality. Although the post is emotionally charged and detailed, it lacks an inquiry or '
    'discussion framing and is primarily a distress vent. Expected responses are emotional '
    'support rather than diverse reasoning, making this a feature–label mismatch rather than '
    'semantic ambiguity.'
)
error_only.iloc[3, error_only.columns.get_loc('error_type')] = 'A'
error_only.iloc[3, error_only.columns.get_loc('error_subtype')] = 'Multi-Intent Ambiguity'
error_only.iloc[3, error_only.columns.get_loc('notes')] = (
    'The post blends personal venting with implicit social commentary, making label assignment '
    'subjective. Reasonable annotators could classify it as a rant (label 0) or a discussion-capable '
    'post (label 1). The disagreement reflects overlapping intents rather than a model failure.'
)
error_only.iloc[4, error_only.columns.get_loc('error_type')] = 'B'
error_only.iloc[4, error_only.columns.get_loc('error_subtype')] = 'Structural Proxy Mismatch'
error_only.iloc[4, error_only.columns.get_loc('notes')] = (
    'The model incorrectly interprets technical detail, external links, and a request for feedback '
    'as signals of discussion quality. In reality, the post is a product showcase or announcement '
    'with limited discussion intent. This reflects a mismatch between structural features and the '
    'label definition.'
)
error_only.iloc[5, error_only.columns.get_loc('error_type')] = 'D'
error_only.iloc[5, error_only.columns.get_loc('error_subtype')] = 'Pragmatic Intent Gap'
error_only.iloc[5, error_only.columns.get_loc('notes')] = (
    'Although the author clearly seeks advice and presents multiple constraints, the request is '
    'embedded narratively rather than framed as a direct question. The model fails to detect '
    'implicit advice-seeking intent, a limitation of shallow text features rather than data quality '
    'or feature weighting.'
)
error_only.iloc[6, error_only.columns.get_loc('error_type')] = 'D'
error_only.iloc[6, error_only.columns.get_loc('error_subtype')] = 'Structural Bias Masking Intent'
error_only.iloc[6, error_only.columns.get_loc('notes')] = (
    'The model downgrades short, title-only speculative questions despite their ability to elicit '
    'creative and varied responses. The error stems from over-penalizing brevity and missing body '
    'content rather than misunderstanding the conceptual nature of the prompt.'
)

error_only.iloc[7, error_only.columns.get_loc('error_type')] = 'B'
error_only.iloc[7, error_only.columns.get_loc('error_subtype')] = 'Structural Proxy Mismatch'
error_only.iloc[7, error_only.columns.get_loc('notes')] = (
    'The model mistakes a long, well-written personal anecdote for a discussion-oriented post. '
    'Despite narrative richness, the post lacks a question or engagement hook, and responses are '
    'likely reactions or jokes. This reflects an overreliance on length and narrative complexity '
    'as quality signals.'
)


In [16]:
error_only

,title,body,code,label,text,title_len,body_len,num_exclamations,questions_per_100_words,has_multiple_paras,...,log_btt_x_has_questions,question_density_per_para,first_person_ratio,true_label,pred_label,correct,model,error_type,error_subtype,notes
51,Living in a foreign country is taking its tow,"I love the country I choose to live, since it ...",,1,Living in a foreign country is taking its tow\...,45,285,0,0.000000,1,...,0.000000,0.0,0.032258,1,0,False,numerical_only,D,Pragmatic Intent Gap,The post is an experiential reflection intende...
34,How would you feel if your country banned Burk...,,,1,How would you feel if your country banned Burk...,73,0,0,7.692308,0,...,-4.304065,1.0,0.000000,1,0,False,numerical_only,D,Structural Bias Masking Intent,The model systematically undervalues title-onl...
45,Can't afford to live and I've exhausted all ot...,So I've been sitting here for the past three d...,,0,Can't afford to live and I've exhausted all ot...,57,333,0,2.702703,0,...,1.750698,2.0,0.137500,0,1,False,numerical_only,B,Engagement Proxy Mismatch,The model overweights emotional intensity and ...
22,I got called a fascist today.,So I was on tiktok and a post came about on a ...,,2,I got called a fascist today.\n\nSo I was on t...,29,531,0,2.564103,0,...,2.875446,3.0,0.095652,2,1,False,numerical_only,A,Multi-Intent Ambiguity,The post blends personal venting with implicit...
74,Add a festive snow effect this Christmas with ...,Hello r/reactjs!\n\nSprinkling some snow acros...,import { SnowOverlay } from 'react-snow-overla...,0,Add a festive snow effect this Christmas with ...,68,502,2,0.000000,1,...,0.000000,0.0,0.037736,0,1,False,numerical_only,B,Structural Proxy Mismatch,The model incorrectly interprets technical det...
66,19F Exhausted New mom not much help from parents.,"Hi i'm a 19f new mom, and I gave birth to my d...",,1,19F Exhausted New mom not much help from paren...,49,1093,0,0.000000,1,...,0.000000,0.0,0.094170,1,0,False,numerical_only,D,Pragmatic Intent Gap,Although the author clearly seeks advice and p...
41,what would the modern day versions of the Gree...,,,1,what would the modern day versions of the Gree...,61,0,0,8.333333,0,...,-4.127134,1.0,0.000000,1,0,False,numerical_only,D,Structural Bias Masking Intent,"The model downgrades short, title-only specula..."
1,I changed a road sign to make my commute easie...,On my daily commute there was very inconvenien...,,0,I changed a road sign to make my commute easie...,61,1185,0,0.000000,1,...,0.000000,0.0,0.065385,0,1,False,numerical_only,B,Structural Proxy Mismatch,"The model mistakes a long, well-written person..."


In [17]:
error_only['error_type'].value_counts()

error_type
D    4
B    3
A    1
Name: count, dtype: int64

In [18]:
error_only['error_subtype'].value_counts()

error_subtype
Pragmatic Intent Gap              2
Structural Bias Masking Intent    2
Structural Proxy Mismatch         2
Engagement Proxy Mismatch         1
Multi-Intent Ambiguity            1
Name: count, dtype: int64

In [19]:
taxonomy_k_fold_analysis = run_k_fold_error_analysis(df)
k_fold_errors = taxonomy_k_fold_analysis['errors_only']
k_fold_errors

,title,body,code,label,text,title_len,body_len,num_exclamations,questions_per_100_words,has_multiple_paras,...,question_density_per_para,first_person_ratio,true_label,pred_label,correct,model,error_type,error_subtype,notes,fold
1,What did you gain by getting out of your home ...,,,2,What did you gain by getting out of your home ...,76,0,0,6.250000,0,...,1.000000,0.000000,2,0,False,numerical_only,,,,0
3,PSA: When you reach out to a co-worker on slac...,I’m going to spend the next 12 minutes distrac...,,0,PSA: When you reach out to a co-worker on slac...,149,403,0,0.000000,1,...,0.000000,0.039604,0,1,False,numerical_only,,,,0
4,Why do some people in 2025 still believe that ...,,,1,Why do some people in 2025 still believe that ...,155,0,0,3.225806,0,...,1.000000,0.060606,1,0,False,numerical_only,,,,0
7,Learn programming as an absolute beginner,I’m a complete outsider here I’ve always wante...,,1,Learn programming as an absolute beginner\n\nI...,41,299,0,5.000000,0,...,3.000000,0.078125,1,2,False,numerical_only,,,,0
8,How would you feel if your country banned Burk...,,,1,How would you feel if your country banned Burk...,73,0,0,7.692308,0,...,1.000000,0.000000,1,0,False,numerical_only,,,,0
10,what would the modern day versions of the Gree...,,,1,what would the modern day versions of the Gree...,61,0,0,8.333333,0,...,1.000000,0.000000,1,0,False,numerical_only,,,,0
11,Where did the girls on Epstein's Island come f...,,,0,Where did the girls on Epstein's Island come f...,96,0,0,18.750000,0,...,3.000000,0.000000,0,2,False,numerical_only,,,,0
12,I graduated today!!!,I graduated yesterday from my university’s Hon...,,1,I graduated today!!!\n\nI graduated yesterday ...,20,345,3,0.000000,1,...,0.000000,0.112676,1,0,False,numerical_only,,,,0
13,CMV: I believe a big majority of social media ...,I think a lot of bad or overtly loud internet ...,,2,CMV: I believe a big majority of social media ...,69,625,0,0.000000,0,...,0.000000,0.024000,2,1,False,numerical_only,,,,0
14,Men's public restrooms are laid out all wrong....,,,2,Men's public restrooms are laid out all wrong....,145,0,0,0.000000,0,...,0.000000,0.000000,2,1,False,numerical_only,,,,0


# Feature Audit: Paragraph-Based Features
During error analysis, the model showed a consistent tendency to overvalue post length and formatting as proxies for discussion quality. In particular, paragraph-based features (e.g., `question_density_per_para`, `has_multiple_paras`) were found to introduce shortcut learning:
* Bullet-point lists and formatting habits artificially inflated paragraph count
* Long narrative or emotional posts were frequently upgraded despite lacking discussion framing
* Short, tightly framed, open-ended questions were downgraded due to minimal body structure

This behavior directly contradicts the labeling guidelines, which explicitly state that ***length and formatting are not indicators of post quality or response entropy.***

To reduce structural bias and shortcut learning, paragraph-related features are intentionally pruned. Only a dampened log-scaled signal is retained (`log_num_para`) to prevent extreme underestimation of effort, without allowing formatting to dominate predictions.

This change prioritizes ***semantic intent and discussion openness*** over superficial structural cues.

In [21]:
df.columns

Index(['title', 'body', 'code', 'label', 'text', 'title_len', 'body_len',
       'num_exclamations', 'questions_per_100_words', 'has_multiple_paras',
       'has_exclamations', 'log_num_para', 'word_count', 'avg_word_len',
       'unique_word_ratio', 'has_code', 'log_btt', 'log_btt_x_has_questions',
       'question_density_per_para', 'first_person_ratio'],
      dtype='object')

In [22]:
df.drop(['question_density_per_para','has_multiple_paras'],axis=1, inplace=True)

# Feature Pruning: Redundant and Shortcut Signals
Several features were identified as either redundant or misaligned with the labeling criteria. In particular:

* Binary threshold features (`has_exclamations`) duplicated continuous counterparts without adding semantic resolution.
* Raw verbosity measures (`word_count`, `log_btt`) promoted expansion and formatting as proxies for post quality, introducing shortcut learning.
* Emotional intensity (`num_exclamations`) showed weak alignment with discussion openness during error analysis.

These features are pruned to reduce structural and stylistic bias. Only interaction-based or dampened variants that reflect ***conditional intent*** (e.g., expansion with a question) are retained.

This pruning step reinforces alignment between model signals and the conceptual labeling framework.

In [24]:
df.columns

Index(['title', 'body', 'code', 'label', 'text', 'title_len', 'body_len',
       'num_exclamations', 'questions_per_100_words', 'has_exclamations',
       'log_num_para', 'word_count', 'avg_word_len', 'unique_word_ratio',
       'has_code', 'log_btt', 'log_btt_x_has_questions', 'first_person_ratio'],
      dtype='object')

In [25]:
df.drop(['has_exclamations','num_exclamations','word_count','log_btt'],axis=1, inplace=True)

In [26]:
df.columns

Index(['title', 'body', 'code', 'label', 'text', 'title_len', 'body_len',
       'questions_per_100_words', 'log_num_para', 'avg_word_len',
       'unique_word_ratio', 'has_code', 'log_btt_x_has_questions',
       'first_person_ratio'],
      dtype='object')

### Retraining and checking errors again, after droping this features and trying to understand now the new pattern

In [28]:
df_train_v2, df_test_v2 = split_df(df,'label')
result_v2 = run_numerical_model(df_train_v2,df_test_v2,'label')
df_train_v2.shape,df_test_v2.shape

((60, 14), (15, 14))

In [29]:
all_result_v2 = result_v2['errors_df']
error_only_v2 = all_result_v2[all_result_v2['correct'] == False].copy()
error_only_v2

,title,body,code,label,text,title_len,body_len,questions_per_100_words,log_num_para,avg_word_len,...,has_code,log_btt_x_has_questions,first_person_ratio,true_label,pred_label,correct,model,error_type,error_subtype,notes
51,Living in a foreign country is taking its tow,"I love the country I choose to live, since it ...",,1,Living in a foreign country is taking its tow\...,45,285,0.000000,1.098612,4.354839,...,0,0.000000,0.032258,1,0,False,numerical_only,,,
34,How would you feel if your country banned Burk...,,,1,How would you feel if your country banned Burk...,73,0,7.692308,0.693147,4.692308,...,0,-4.304065,0.000000,1,0,False,numerical_only,,,
45,Can't afford to live and I've exhausted all ot...,So I've been sitting here for the past three d...,,0,Can't afford to live and I've exhausted all ot...,57,333,2.702703,0.693147,4.297297,...,0,1.750698,0.137500,0,1,False,numerical_only,,,
14,Stripe Native crashes on google pixel device,I’m integrating Stripe PaymentSheet (React Nat...,const openPaymentSheet = async () => {\n try ...,2,Stripe Native crashes on google pixel device\n...,44,331,0.000000,1.945910,5.266667,...,1,0.000000,0.051724,2,0,False,numerical_only,,,
38,How to stop avoiding things,When something I have to do feels complicated ...,,1,How to stop avoiding things\n\nWhen something ...,27,619,0.813008,0.693147,4.268293,...,0,3.097515,0.061069,1,2,False,numerical_only,,,
22,I got called a fascist today.,So I was on tiktok and a post came about on a ...,,2,I got called a fascist today.\n\nSo I was on t...,29,531,2.564103,0.693147,3.803419,...,0,2.875446,0.095652,2,1,False,numerical_only,,,
63,How do some men get so many women when they ar...,"I feel like we've all seen it. Like, the guy w...",,2,How do some men get so many women when they ar...,86,415,3.157895,1.098612,4.294737,...,0,1.564777,0.049020,2,1,False,numerical_only,,,
41,what would the modern day versions of the Gree...,,,1,what would the modern day versions of the Gree...,61,0,8.333333,0.693147,4.166667,...,0,-4.127134,0.000000,1,0,False,numerical_only,,,
1,I changed a road sign to make my commute easie...,On my daily commute there was very inconvenien...,,0,I changed a road sign to make my commute easie...,61,1185,0.000000,1.791759,3.894118,...,0,0.000000,0.065385,0,1,False,numerical_only,,,


# Experiment Conclusion & Direction Change

The current modeling approach attempted to predict a single post-quality label (0/1/2) using structural and lexical features.

Error taxonomy analysis revealed that the majority of persistent errors stem from representational ambiguity rather than feature insufficiency. Specifically:

- Short, tightly framed posts with high discussion potential were systematically downgraded.
- Long or multi-paragraph narrative posts were systematically upgraded despite low discussion openness.

These patterns were stable across feature ablations and indicate that the labeling decision implicitly combines two distinct latent dimensions:

1. **Author Effort** — the amount of contextual and cognitive work performed by the author.
2. **Discussion Openness** — the likelihood of diverse, non-convergent responses.

Attempting to learn both dimensions with a single scalar label forces the model to rely on brittle structural proxies (e.g., paragraph count), leading to shortcut learning.

As a result, the project will transition to a two-axis formulation, with separate models for **Effort** and **Openness**. The original label will be derived from these axes rather than predicted directly.

This change represents a **problem reformulation**, not a feature iteration.
